In [1]:
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, roc_curve, precision_recall_curve, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from scipy.stats import t, ttest_rel
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

## Load data

In [ ]:
data_path = '/data/gas_new/values_refine/RGB_diff_refine_label_new.csv'

diff = pd.read_csv(data_path, index_col='id')
diff_n2 = (np.sqrt(diff.iloc[:, :-1]**2)).T.groupby(np.arange(diff.shape[1]-1) // 3).sum().T

X = diff.drop('label', axis=1, inplace=False).to_numpy()
X_n2 = diff_n2.to_numpy()
y = diff['label'].to_numpy().astype(int)
print(len(y), sum(y))

(134, np.int64(81))

## Model Training

In [ ]:
acc, auc, f1, balanced_acc, precision, recall = [], [], [], [], [], []
N_SPLITS=5
y_pred = []
score = []
all_fpr = []
all_tpr = []

cv = StratifiedKFold(n_splits=N_SPLITS, 
                     shuffle=True, 
                     random_state=42
                     )

for fold, (train_idx, val_idx) in enumerate(cv.split(X_n2, y)):

    print(f"--- Fold {fold} ---")

    X_train, X_val = X_n2[train_idx], X_n2[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # --- SMOTE ---
    smote = SMOTE(random_state=42)
    X_train, y_train = smote.fit_resample(X_train, y_train)

    model = XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',  
        n_estimators=300,       
        learning_rate=0.1,
        max_depth=4,
        min_child_weight=1,
        gamma=0,
        reg_alpha=0.0,
        reg_lambda=1
        )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)  # 离散类别 {0,1}
    y_prob = model.predict_proba(X_val)[:, 1]  # 概率

    acc.append(accuracy_score(y_val, y_pred))
    auc.append(roc_auc_score(y_val, y_prob))
    f1.append(f1_score(y_val, y_pred))
    balanced_acc.append(balanced_accuracy_score(y_val, y_pred))
    precision.append(precision_score(y_val, y_pred))
    recall.append(recall_score(y_val, y_pred))
    fpr, tpr, thresholds = roc_curve(y_val, y_prob)
    all_fpr.append(fpr)
    all_tpr.append(tpr)

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

for i in range(5):
    ax[i].plot(all_fpr[i], all_tpr[i], color='darkorange', lw=2, label=f'ROC curve (AUC = {auc[i]:.3f})')
    ax[i].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
    ax[i].set_title(f'Fold {i+1}', fontsize=14)
    ax[i].set_xlabel('False Positive Rate', fontsize=14)
    ax[i].legend(loc='lower right')
    # ax[i].grid(True, linestyle='--', alpha=0.5)

# plt.legend(loc='lower right')
fig.suptitle('ROC Curve', fontsize=14, fontweight='bold')
fig.supylabel('True positive rate', fontsize=14)
plt.tight_layout(rect=[0.005, -0.05, 1, 0.999])
plt.show()

In [ ]:
print(f"===== Overall Results ({N_SPLITS}-Fold Avg) =====")
print(f"ACC={np.mean(acc):.3f} ± {np.std(acc):.3f}")
print(f"F1 ={np.mean(f1):.3f} ± {np.std(f1):.3f}")
print(f"AUC={np.mean(auc):.3f} ± {np.std(auc):.3f}")
print(f"Precision={np.mean(precision):.3f} ± {np.std(precision):.3f}")
print(f"Recall={np.mean(recall):.3f} ± {np.std(recall):.3f}")

In [11]:
import json

js_val = {}
js_val['threshold'] = 0.5
js_val['accuracy'] = acc
js_val['f1_score'] = f1
js_val['auc'] = auc
js_val['precision'] = precision
js_val['recall'] = recall
js_val['fpr'] = [fpr.tolist() for fpr in all_fpr]
js_val['tpr'] = [tpr.tolist() for tpr in all_tpr]

with open('weights/RGB_diff_refine_label_new_n2/xgboost_val.json', 'w') as f:
    json.dump(js_val, f)